# Usings

In [65]:
from typing import List
from langchain_openai import AzureChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate, ChatPromptTemplate, FewShotChatMessagePromptTemplate
from dotenv import load_dotenv
import os

In [66]:
load_dotenv()

True

# Classe ChatBot

In [74]:
class ChatBot:
    def __init__(self,
                 name:str,
                 instructions:str,
                 examples: List[dict],
                 model:str="gpt-4o-mini", 
                 temperature:float=0.0):
        
        self.name = name

        model_name = os.getenv("MODEL_NAME")
        if not model_name:
            raise ValueError("Please set the MODEL_NAME environment variable.")
        
        self.llm = AzureChatOpenAI(
            azure_deployment=model_name,
            api_version="2024-12-01-preview",
            temperature=0.0,
        )
        
        example_prompt = ChatPromptTemplate.from_messages(
            [
                ("system", instructions),
                ("human", "{input}"),
                ("ai", "{output}"),
            ]
        )
        prompt_template = FewShotChatMessagePromptTemplate(
            example_prompt=example_prompt,
            examples=examples,
        )

        # Memory
        self.messages = prompt_template.invoke({}).to_messages()

    def invoke(self, user_message:str)->AIMessage:
        # 1. Adiciona a mensagem do usuário ao histórico
        self.messages.append(HumanMessage(content=user_message))

        # 2. Chama o modelo passando todo o histórico
        ai_message = self.llm.invoke(self.messages)

        # 3. Adiciona a resposta da IA ao histórico (mantém a memória)
        self.messages.append(ai_message)

        return ai_message

# Instruçoes - perfil da LLM

In [ ]:
instructions = (
    "Você é um professor de inteligencia artificial, "
    "especialista em LLM, MCP, RAG, LangChain, LangGraph, LangFuse e Copilot Studio. "
    "Sabe perfeitamente como integrar com ferramentas externas como workday e servicenow. "
    "Use sempre respostas bem estruturadas, com exemplos e referências. Não invente informações, se não souber, diga que não sabe. "
    "Evite responder com siglas cujo significado não seja claro. "
)

# Exemplos esperados - Serve para treinar formato de resposta

In [ ]:
examples = [
    {
        "input": "o que é MCP?", 
        "output": "{pergunta\": \"O que é MCP?\", \"resposta\": \"O MCP (Model Context Protocol) é um padrão aberto que padroniza a forma como sistemas de inteligência artificial se conectam a fontes de dados e ferramentas externas\"}"
    },
    
    {
        "input": "O que é RAG?", 
        "output": "{\"pergunta\": \"O que é RAG?\",\"resposta\": \"RAG (Retrieval-Augmented Generation) é uma técnica de processamento de linguagem natural que combina a recuperação de informações com a geração de texto.\"}"
    },
]

# Instancia do teacher

In [77]:
teacher = ChatBot(
    name="Teacher",
    instructions=instructions,
    examples=examples
)

# Chamadas

In [ ]:
teacher.invoke("O que é LangChain?")

AIMessage(content='{"nome": "Gui","idade": 6,"cep": "12234-567"}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 183, 'total_tokens': 206, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 3, 'engine_ttft_ms': 48, 'engine_ttlt_ms': 118, 'pre_inference_ms': 211, 'service_tbt_ms': 2, 'service_ttft_ms': 469, 'service_ttlt_ms': 517, 'user_visible_ttft_ms': 258}}, 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EKqIxYPVm5M010xdco4LqFMp9b0Lk', 'service_tier': 'default', 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'detected': False, 'filtered': False}, 'self_harm': {'filtered': Fa

In [72]:
teacher.invoke("O que é LangSmith?")

AIMessage(content='**LangSmith** é uma plataforma para **observabilidade, avaliação e depuração** de aplicações baseadas em modelos de linguagem.\n\n### Em termos simples\nEle serve para você **acompanhar o que a aplicação está fazendo**, entender **por que uma resposta saiu errada** e **medir a qualidade** dos resultados.\n\n### Principais usos\n- **Tracing**: ver o passo a passo de uma chamada do sistema\n- **Debugging**: identificar onde um fluxo falhou\n- **Avaliação**: testar qualidade de respostas com conjuntos de dados\n- **Monitoramento**: acompanhar desempenho em produção\n- **Prompt management**: versionar e comparar prompts\n\n### Exemplo prático\nSe você tem um chatbot com:\n1. busca em documentos\n2. chamada para uma ferramenta externa\n3. geração da resposta final\n\nO LangSmith permite ver:\n- quais documentos foram recuperados\n- quanto tempo cada etapa levou\n- qual prompt foi enviado ao modelo\n- qual resposta foi gerada\n\n### Relação com LangChain\nO LangSmith é da 

In [73]:
teacher.invoke("O que é o FewShotChatMessagePromptTemplate ? porque preciso definir um?")

AIMessage(content='O **FewShotChatMessagePromptTemplate** é uma classe do **LangChain** usada para montar prompts de chat com **exemplos de entrada e saída** antes da pergunta real do usuário.\n\n## O que ele faz\nEle permite inserir no prompt alguns exemplos do tipo:\n\n- **Pergunta do usuário**\n- **Resposta esperada**\n\nIsso ajuda o modelo a entender melhor o padrão de resposta que você quer.\n\n## Por que usar\nVocê define um **few-shot prompt** quando quer que o modelo:\n- siga um **formato específico**\n- responda com **mais consistência**\n- aprenda o **estilo** ou a **estrutura** desejada sem precisar treinar o modelo\n\n## Exemplo simples\nSe você quer que o assistente responda sempre de forma objetiva:\n\n**Exemplo 1**\n- Usuário: “O que é RAG?”\n- Assistente: “RAG é uma técnica que combina recuperação de माहिती com geração de texto.”\n\n**Exemplo 2**\n- Usuário: “O que é LangChain?”\n- Assistente: “LangChain é um framework para construir aplicações com modelos de linguagem.